In [ ]:
Обобщить - даатсеты с разными 

In [ ]:
import torch
import random
from tqdm import tqdm

# =========================
# Config
# =========================
N_SAMPLES = 100
DEPTH = 3
MAX_LEN = 16
SAVE_PATH = r"C:\Users\ivan\WORK\workshops\NeurIPS\fraca\listops_train_fixed.pt"
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# Vocabulary (FIXED)
# =========================
VOCAB = ['[PAD]', '[CLS]', '[', ']', 'MAX', 'MIN', 'SUM'] + [str(i) for i in range(10)]

vocab = {tok: i for i, tok in enumerate(VOCAB)}
inv_vocab = {i: tok for tok, i in vocab.items()}

PAD_ID = vocab['[PAD]']
CLS_ID = vocab['[CLS]']

OPS = ['MAX', 'MIN', 'SUM']

# =========================
# Expression generation
# =========================
def generate_tree(depth=3):
    if depth == 0:
        return str(random.randint(0, 9))

    op = random.choice(OPS)
    num_args = random.randint(2, 4)
    args = [generate_tree(depth - 1) for _ in range(num_args)]
    return f"[{op} {' '.join(args)}]"

# =========================
# Expression evaluation
# =========================
def parse_expr(tokens, pos=0):
    tok = tokens[pos]

    if tok != '[':
        return int(tok), pos + 1

    op = tokens[pos + 1]
    pos = pos + 2
    args = []

    while tokens[pos] != ']':
        val, pos = parse_expr(tokens, pos)
        args.append(val)

    pos += 1

    if op == 'MAX':
        return max(args), pos
    elif op == 'MIN':
        return min(args), pos
    elif op == 'SUM':
        return sum(args), pos
    else:
        raise ValueError(f"Unknown op: {op}")

def eval_tree(expr: str) -> int:
    tokens = expr.replace('[', ' [ ').replace(']', ' ] ').split()
    value, end_pos = parse_expr(tokens, 0)
    if end_pos != len(tokens):
        raise ValueError("Parsing did not consume all tokens.")
    return value

# =========================
# Tokenization (FIXED)
# =========================
def tokenize(expr, max_len=128):
    tokens = ['[CLS]'] + expr.replace('[', ' [ ').replace(']', ' ] ').split()
    ids = [vocab[t] for t in tokens]

    if len(ids) < max_len:
        ids += [PAD_ID] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return torch.tensor(ids, dtype=torch.long)

# =========================
# Dataset generation
# =========================
def generate_dataset(n_samples=10000, depth=3, max_len=128):
    X, Y = [], []

    for _ in tqdm(range(n_samples), desc="Generating dataset"):
        expr = generate_tree(depth=depth)
        y = eval_tree(expr) % 10   # фиксируем классы 0..9
        x = tokenize(expr, max_len=max_len)

        X.append(x)
        Y.append(y)

    X = torch.stack(X)
    Y = torch.tensor(Y, dtype=torch.long)
    return X, Y

# =========================
# Decode (FIXED)
# =========================
def decode(x_tensor):
    toks = []
    for idx in x_tensor.tolist():
        tok = inv_vocab.get(int(idx), '?')
        if tok != '[PAD]':
            toks.append(tok)
    return " ".join(toks)

# =========================
# Run
# =========================
X, Y = generate_dataset(
    n_samples=N_SAMPLES,
    depth=DEPTH,
    max_len=MAX_LEN
)

print("Dataset created")
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Y min:", Y.min().item())
print("Y max:", Y.max().item())
print("Unique classes:", sorted(Y.unique().tolist()))

print("\nExample 1")
print("Decoded:", decode(X[0]))
print("Target :", Y[0].item())

print("\nExample 2")
print("Decoded:", decode(X[1]))
print("Target :", Y[1].item())

torch.save({'X': X, 'Y': Y}, SAVE_PATH)
print(f"\nSaved to: {SAVE_PATH}")

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import time
import os
import json

# ========= OPTIONAL: Lion =========
try:
    from lion_pytorch import Lion
    HAS_LION = True
except:
    HAS_LION = False

# ========= CONFIG =========
RUNS = 5
EPOCHS = 15
VAL_SPLIT = 0.2
LOG_ROOT = "logs_"
os.makedirs(LOG_ROOT, exist_ok=True)

# ========= Dataset =========
class ListOpsDataset(Dataset):
    def __init__(self, path):
        data = torch.load(path)
        self.X = data['X']
        self.Y = data['Y'].long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.Y[i]

# ========= Model =========
class TransformerModel(nn.Module):
    def __init__(self, vocab_size=32, d_model=128, nhead=4, num_layers=2, num_classes=10):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.fc(x)

# ========= Optimizers =========
def get_optimizer(name, model):
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=3e-4)
    elif name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=3e-4)
    elif name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
    elif name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=1e-3)
    elif name == "adagrad":
        return torch.optim.Adagrad(model.parameters(), lr=1e-2)
    elif name == "lion" and HAS_LION:
        return Lion(model.parameters(), lr=3e-4)
    else:
        raise ValueError(name)

# ========= Logging =========
def log(msg, path):
    print(msg)
    with open(path, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

# ========= F1 =========
def compute_macro_f1(preds, targets, num_classes):
    f1_scores = []
    preds = preds.cpu()
    targets = targets.cpu()

    for c in range(num_classes):
        tp = ((preds == c) & (targets == c)).sum().item()
        fp = ((preds == c) & (targets != c)).sum().item()
        fn = ((preds != c) & (targets == c)).sum().item()

        if tp == 0 and (fp == 0 or fn == 0):
            f1_scores.append(0.0)
            continue

        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)

        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        f1_scores.append(f1)

    return sum(f1_scores) / len(f1_scores)

# ========= Setup =========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ListOpsDataset(r"C:\Users\ivan\WORK\workshops\NeurIPS\fraca\listops_train_fixed.pt")
num_classes = int(dataset.Y.max().item()) + 1

# split
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

criterion = nn.CrossEntropyLoss()

optimizers = ["adam", "adamw", "sgd", "rmsprop", "adagrad"]
if HAS_LION:
    optimizers.append("lion")

# ========= VALIDATION =========
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_targets = []
    total_conf = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            loss = criterion(out, y)
            total_loss += loss.item()

            probs = torch.softmax(out, dim=1)
            total_conf += probs.max(dim=1)[0].mean().item()

            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

            all_preds.append(preds)
            all_targets.append(y)

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    f1 = compute_macro_f1(all_preds, all_targets, num_classes)

    return (
        total_loss / len(loader),
        correct / total,
        f1,
        total_conf / len(loader)
    )

# ========= TRAIN =========
def train_one(opt_name, run_id):

    opt_dir = os.path.join(LOG_ROOT, opt_name)
    os.makedirs(opt_dir, exist_ok=True)

    log_path = os.path.join(opt_dir, f"run_{run_id}.txt")
    hist_path = os.path.join(opt_dir, f"run_{run_id}_history.json")

    model = TransformerModel(num_classes=num_classes).to(device)
    optimizer = get_optimizer(opt_name, model)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
        "train_confidence": [],
        "val_confidence": [],
        "grad_norm": [],
        "epoch_time": []
    }

    best_val_acc = 0
    best_val_loss = float("inf")

    run_start = time.time()

    log(f"\n=== {opt_name.upper()} | RUN {run_id} ===", log_path)

    for epoch in range(EPOCHS):
        start_time = time.time()

        model.train()
        total_loss = 0
        correct = 0
        total = 0
        total_conf = 0
        grad_norm_epoch = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)

            loss.backward()

            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            grad_norm_epoch += total_norm ** 0.5

            optimizer.step()

            total_loss += loss.item()

            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            probs = torch.softmax(out, dim=1)
            total_conf += probs.max(dim=1)[0].mean().item()

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        train_conf = total_conf / len(train_loader)
        grad_norm = grad_norm_epoch / len(train_loader)

        val_loss, val_acc, val_f1, val_conf = evaluate(model, val_loader)

        best_val_acc = max(best_val_acc, val_acc)
        best_val_loss = min(best_val_loss, val_loss)

        epoch_time = time.time() - start_time

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)
        history["train_confidence"].append(train_conf)
        history["val_confidence"].append(val_conf)
        history["grad_norm"].append(grad_norm)
        history["epoch_time"].append(epoch_time)

        log(
            f"Epoch {epoch+1} | "
            f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.3f} | "
            f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.3f} | "
            f"Val F1 {val_f1:.3f} | "
            f"Train Conf {train_conf:.3f} | Val Conf {val_conf:.3f} | "
            f"Grad {grad_norm:.3f} | Time {epoch_time:.2f}s",
            log_path
        )

    total_time = time.time() - run_start

    with open(hist_path, "w") as f:
        json.dump(history, f, indent=4)

    log(f"\nBest Val Acc: {best_val_acc:.4f}", log_path)
    log(f"Best Val Loss: {best_val_loss:.4f}", log_path)
    log(f"TOTAL RUN TIME: {total_time:.2f} sec", log_path)

# ========= RUN ALL =========
for opt in optimizers:
    for run in range(RUNS):
        train_one(opt, run)

print("\nALL DONE")

/home/tahiti/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ivan\\WORK\\workshops\\NeurIPS\\fraca\\listops_train_fixed.pt'